In [17]:
import base64
import os
from typing import List, Optional, Literal
import tqdm
import re
from pathlib import Path

import cv2
from tenacity import retry, stop_after_attempt, wait_random_exponential

In [19]:
open_ai_scene_types = {
    ("apartment",): "Apartment",
    ("bathroom",): "Bathroom",
    ("bedroom", "hotel"): "Bedroom / Hotel",
    ("hotel", "bedroom"): "Bedroom / Hotel",
    ("bookstore", "library"): "Bookstore / Library",
    ("library", "bookstore"): "Bookstore / Library",
    ("classroom",): "Classroom",
    ("conference", "room"): "Conference Room",
    ("copy", "mail", "room"): "Copy / Mail Room",
    ("mail", "copy", "room"): "Copy / Mail Room",
    ("kitchen",): "Kitchen",
    ("laundry",): "Laundry Room",
    ("living", "room", "lounge"): "Living room / Lounge",
    ("lounge", "living", "room"): "Living room / Lounge",
    ("lobby",): "Lobby",
    ("office",): "Office",
    ("storage", "basement", "garage"): "Storage / Basement / Garage",
    ("storage", "garage", "basement"): "Storage / Basement / Garage",
    ("basement", "storage", "garage"): "Storage / Basement / Garage",
    ("basement", "garage", "storage"): "Storage / Basement / Garage",
    ("garage", "storage", "basement"): "Storage / Basement / Garage",
    ("garage", "basement", "storage"): "Storage / Basement / Garage",
}

In [2]:
def prepare_messages(content: str, reasoning: bool = False):
    messages = []
    messages.append({"role": "user", "content": content})
    return messages

In [3]:
def prepare_vision_messages(
    prefix: Optional[str] = None,
    suffix: Optional[str] = None,
    image_paths: Optional[List[str]] = None,
    image_size: Optional[int] = 512,
):
    messages = []
   
    if image_paths is None:
        image_paths = []

    content = []

    for path in image_paths:
        frame = cv2.imread(path)
        if image_size:
            factor = image_size / max(frame.shape[:2])
            frame = cv2.resize(frame, dsize=None, fx=factor, fy=factor)
        _, buffer = cv2.imencode(".png", frame)
        frame = base64.b64encode(buffer).decode("utf-8")
        content.append(
            {
                "image": f"data:image/png;base64,{frame}",
                "type": "image",
            }
        )

    text = []
    if prefix:
        text.append(prefix)
    if suffix:
        text.append(suffix)
    
    text = "\n\n".join(text)

    content.append({"text": text, "type": "text"})
    messages.append({"role": "user", "content": content})

    return messages

In [4]:
DEFAULT_DATA_DIR: Path = Path("./") / "prompts_sceneType"

PROMPT_NAME_TO_PATH = {
    "blind": DEFAULT_DATA_DIR / Path("blind.txt"),
    "blind_not_step_by_step": DEFAULT_DATA_DIR / Path("blind_not_step_by_step.txt"),
    "vision": DEFAULT_DATA_DIR / Path("vision.txt"),
    "vision_not_step_by_step": DEFAULT_DATA_DIR / Path("vision_not_step_by_step.txt"),
    "vision_and_text_prefix": DEFAULT_DATA_DIR / Path("vision_and_text_prefix.txt"),
    "vision_and_text_prefix_not_step_by_step": DEFAULT_DATA_DIR / Path("vision_and_text_prefix_not_step_by_step.txt"),
    "vision_and_text_suffix": DEFAULT_DATA_DIR / Path("vision_and_text_suffix.txt"),
}

def load_prompt(name: str):
    if name not in PROMPT_NAME_TO_PATH:
        raise ValueError("invalid prompt: {}".format(name))
    path = PROMPT_NAME_TO_PATH[name]
    with path.open("r") as f:
        return f.read().strip()

In [5]:
import json
# load dataset
dataset = {}
for item in json.load(open("data/open-eqa-v0_sceneType.json", "r", encoding="utf-8")):
    if "sceneType" in item:
        episode_history = item["episode_history"]
        if not episode_history in dataset:
            dataset[episode_history] = {"questions":[], "answers":[], "sceneType": item["sceneType"]}
        dataset[episode_history]["questions"].append(item["question"])
        dataset[episode_history]["answers"].append(item["answer"])
print("found {:,} episode histories".format(len(dataset)))

found 79 episode histories


In [30]:
model_name = "Qwen/Qwen2.5-VL-7B-Instruct"
seed = 1234
max_tokens = 4000
temperature = 0.2
image_size = 512
num_q_and_a = 14
prompts = ["blind", "blind_not_step_by_step"]
if "vision" in model_name or "modal" in model_name or "VL" in model_name:
    prompts.extend(["vision", "vision_not_step_by_step", "vision_and_text", "vision_and_text_not_step_by_step"])

In [31]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info
import torch

# default: Load the model on the available device(s)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    device_map="auto",
)

# default processer
processor = AutoProcessor.from_pretrained(model_name, use_fast=True)

Loading checkpoint shards: 100%|██████████| 5/5 [00:03<00:00,  1.43it/s]


In [32]:
for prompt in prompts:
    results = []
    freq = {}
    for idx, (episode_history, item) in enumerate(tqdm.tqdm(list(dataset.items()))):
        # get Q&A
        questions_and_answers = []
        for question, answer in zip(item["questions"], item["answers"]):
            questions_and_answers.append(f"Q: {question}\nA: {answer}")
            if len(questions_and_answers) == num_q_and_a:
                break
        questions_and_answers = "\n\n".join(questions_and_answers)
        
        # extract scene paths and prepare the prompt
        paths = None
        if "vision" in prompt:
            image_paths = [os.path.join("data/scene_images", f"{episode_history}.png")]
            
            suffix = None
            if "text" in prompt:
                if "not_step_by_step" in prompt:
                    prefix = load_prompt("vision_and_text_prefix_not_step_by_step")
                else:
                    prefix = load_prompt("vision_and_text_prefix")
                suffix = load_prompt("vision_and_text_suffix")
                suffix = suffix.format(questions_and_answers=questions_and_answers)            
            else:
                if "not_step_by_step" in prompt:
                    prefix = load_prompt("vision_not_step_by_step")
                else:
                    prefix = load_prompt("vision")
    
            messages = prepare_vision_messages(
                prefix=prefix, suffix=suffix, image_paths=image_paths, image_size=image_size
            )
        else:
            prompt_text = load_prompt(prompt)
            messages = prepare_messages(prompt_text.format(questions_and_answers=questions_and_answers))
        
        # Preparation for inference
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        inputs = inputs.to("cuda")
        
        # Inference: Generation of the output
        torch.manual_seed(seed)
        generated_ids = model.generate(**inputs, max_new_tokens=max_tokens, temperature=temperature)
        generated_ids_trimmed = [
            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        lower_output = output[0].lower()
        if "the category is" in lower_output:
            lower_output = re.sub(".*the category is", "", lower_output, flags=re.MULTILINE | re.DOTALL)
            words = [re.sub("\W+", "", word) for word in lower_output.split()]
            words = [word for word in words if word and word != "or"]
        else:
            words = [re.sub("\W+", "", word) for word in lower_output.split()]
            words = [word for word in words if word and word != "or"]
        estimated_sceneType = None
        for i in range(len(words)):
            for j in range(1, 4):
                if i + j == len(words) + 1:
                    break
                scene_type_key = tuple(words[i:i+j])
                if scene_type_key in open_ai_scene_types:
                    estimated_sceneType = open_ai_scene_types[scene_type_key]
                    break
            if estimated_sceneType:
                break
        
        # count sceneTypes
        if not item["sceneType"] in freq:
            freq[item["sceneType"]] = [0, 0]
        freq[item["sceneType"]][0] += 1
        if item["sceneType"] == estimated_sceneType:
            freq[item["sceneType"]][1] += 1

        # store results
        results.append({
            "episode_history": episode_history,
            "output": output,
            "estimated_sceneType": estimated_sceneType,
            "sceneType": item["sceneType"]
        })
        json.dump(results, open("data/results/{}-{}-{}.json".format(re.sub(r".*/", "", model_name), prompt, seed), "w"), indent=2)

    # save at end (redundant)
    json.dump(results, open("data/results/{}-{}-{}.json".format(re.sub(r".*/", "", model_name), prompt, seed), "w"), indent=2)
    print("saving {:,} answers".format(len(results)))
    
    for k,v in freq.items():
        print(k, v)
        

100%|██████████| 79/79 [11:52<00:00,  9.02s/it]


saving 79 answers
Bookstore / Library [3, 0]
Office [12, 7]
Bedroom / Hotel [18, 14]
Bathroom [8, 7]
Living room / Lounge [8, 6]
Apartment [4, 0]
Conference Room [9, 4]
Copy / Mail Room [4, 1]
Kitchen [5, 4]
Laundry Room [1, 1]
Storage / Basement / Garage [2, 0]
Lobby [4, 0]
Classroom [1, 1]


100%|██████████| 79/79 [06:33<00:00,  4.99s/it]


saving 79 answers
Bookstore / Library [3, 2]
Office [12, 6]
Bedroom / Hotel [18, 12]
Bathroom [8, 8]
Living room / Lounge [8, 7]
Apartment [4, 0]
Conference Room [9, 7]
Copy / Mail Room [4, 4]
Kitchen [5, 4]
Laundry Room [1, 1]
Storage / Basement / Garage [2, 0]
Lobby [4, 0]
Classroom [1, 1]


100%|██████████| 79/79 [02:07<00:00,  1.61s/it]


...

saving 79 answers
Bookstore / Library [3, 1]
Office [12, 11]
Bedroom / Hotel [18, 10]
Bathroom [8, 7]
Living room / Lounge [8, 5]
Apartment [4, 0]
Conference Room [9, 6]
Copy / Mail Room [4, 0]
Kitchen [5, 4]
Laundry Room [1, 1]
Storage / Basement / Garage [2, 0]
Lobby [4, 0]
Classroom [1, 1]


100%|██████████| 79/79 [09:36<00:00,  7.29s/it]


saving 79 answers
Bookstore / Library [3, 1]
Office [12, 11]
Bedroom / Hotel [18, 10]
Bathroom [8, 7]
Living room / Lounge [8, 7]
Apartment [4, 0]
Conference Room [9, 6]
Copy / Mail Room [4, 2]
Kitchen [5, 4]
Laundry Room [1, 1]
Storage / Basement / Garage [2, 1]
Lobby [4, 1]
Classroom [1, 1]


100%|██████████| 79/79 [03:17<00:00,  2.50s/it]

saving 79 answers
Bookstore / Library [3, 2]
Office [12, 8]
Bedroom / Hotel [18, 12]
Bathroom [8, 7]
Living room / Lounge [8, 6]
Apartment [4, 0]
Conference Room [9, 6]
Copy / Mail Room [4, 2]
Kitchen [5, 4]
Laundry Room [1, 1]
Storage / Basement / Garage [2, 1]
Lobby [4, 1]
Classroom [1, 1]


In [15]:
output

['The category is (Library)']